# Seed experiment analysis

This notebook measures model performance and reproducibility across random seeds. It mirrors `seed_experiment_analysis.py`, while keeping each step visible for inspection and adaptation.

The default input is `results/experiments.csv`; all generated tables and figures are saved under `results/analysis/seed/`.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

## 1. Setup

Run the notebook from either the project root or the `analysis/` directory. No machine-specific absolute paths are stored.

In [3]:
from pathlib import Path
import sys

start = Path.cwd()
candidates = [start, *start.parents]

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "." / "analysis" / "seed_experiment_analysis.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "找不到 results/analysis/seed_experiment_analysis.py，"
        "请确认文件已复制到 rxrx1 主项目中。"
    )

SCRIPT_DIR = PROJECT_ROOT / "analysis"
sys.path.insert(0, str(SCRIPT_DIR))

from analysis.seed_experiment_analysis import (
    add_train_val_gaps,
    filter_seed_experiments,
    load_experiments,
    plot_seed_analysis,
    select_median_seed,
    summarize_by_group,
)

CSV_PATH = PROJECT_ROOT / "results" / "experiments.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "analysis" / "seed"

print("项目根目录：", PROJECT_ROOT)
print("分析脚本：", SCRIPT_DIR / "seed_experiment_analysis.py")
print("输入 CSV：", CSV_PATH)

项目根目录： C:\rxrx1
分析脚本： C:\rxrx1\analysis\seed_experiment_analysis.py
输入 CSV： C:\rxrx1\results\experiments.csv


## 2. Parameters

Change `INPUT_RELATIVE` if the experiment table has a different project-relative location. `GROUP_PATTERN` is a case-insensitive regular expression applied to `experiment_group`.

In [4]:
INPUT_RELATIVE = Path("results/experiments.csv")
OUTPUT_RELATIVE = Path("results/analysis/seed")
GROUP_PATTERN = "seed"

CSV_PATH = PROJECT_ROOT / INPUT_RELATIVE
OUTPUT_DIR = PROJECT_ROOT / OUTPUT_RELATIVE
CSV_PATH, OUTPUT_DIR

(WindowsPath('C:/rxrx1/results/experiments.csv'),
 WindowsPath('C:/rxrx1/results/analysis/seed'))

## 3. Load and filter runs

The loader validates the expected schema and converts numeric columns. The filter keeps only experiment groups matching `GROUP_PATTERN`.

In [5]:
experiments = load_experiments(CSV_PATH)
seed_runs = filter_seed_experiments(experiments, GROUP_PATTERN)

print(f"All rows: {len(experiments):,}")
print(f"Seed rows: {len(seed_runs):,}")
print(f"Seed groups: {seed_runs['experiment_group'].nunique():,}")
display(seed_runs.head())

All rows: 8
Seed rows: 5
Seed groups: 1


,experiment_id,experiment_name,experiment_group,model,seed,image_size,batch_size,epochs,optimizer,final_train_acc,...,final_val_loss,best_epoch,best_train_acc,best_train_loss,best_val_acc,best_val_loss,runtime_seconds,runtime_minutes,runtime_per_epoch_minutes,note
0,exp001,resnet18_seed0_size256,seed,resnet18,0,256,32,5,adamw,0.9806,...,5.6588,4,0.9144,3.5154,0.0262,5.6918,2076,34.60,6.92,seed comparison
1,exp002,resnet18_seed42_size256,seed,resnet18,42,256,32,5,adamw,0.9911,...,5.6385,4,0.9350,3.5056,0.0269,5.6584,2074,34.57,6.91,seed comparison; reference seed
2,exp003,resnet18_seed2026_size256,seed,resnet18,2026,256,32,5,adamw,0.9819,...,5.6487,4,0.8958,3.6189,0.0213,5.6625,1757,29.28,5.86,seed comparison
3,exp004,resnet18_seed2386_size256,seed,resnet18,2386,256,32,5,adamw,0.9869,...,5.6579,4,0.9164,3.5621,0.0225,5.6621,1757,29.28,5.86,seed comparison
4,exp005,resnet18_seed3407_size256,seed,resnet18,3407,256,32,5,adamw,0.9914,...,5.6625,4,0.9231,3.4887,0.0275,5.6698,1739,28.98,5.80,best seed result at size256


## 4. Add train-validation gaps

Accuracy gaps are `train − validation`; loss gaps are `validation − train`. Therefore, a positive gap consistently means validation performance is worse. The final gap compares values from the same final epoch. A best-value gap may compare optima reached at different epochs, so interpret it as descriptive rather than as a same-checkpoint generalization gap.

In [6]:
seed_runs = add_train_val_gaps(seed_runs)
gap_columns = [
    "experiment_group",
    "seed",
    "final_acc_train_val_gap",
    "best_acc_train_val_gap",
    "final_loss_val_train_gap",
    "best_loss_val_train_gap",
]
display(seed_runs[gap_columns])

,experiment_group,seed,final_acc_train_val_gap,best_acc_train_val_gap,final_loss_val_train_gap,best_loss_val_train_gap
0,seed,0,0.9593,0.8882,3.0275,2.1764
1,seed,42,0.9667,0.9081,2.9892,2.1528
2,seed,2026,0.9619,0.8745,2.9094,2.0436
3,seed,2386,0.9669,0.8939,2.9767,2.1000
4,seed,3407,0.9676,0.8956,3.0295,2.1811


## 5. Summary statistics

Each metric is summarized by experiment group with mean, sample standard deviation (`ddof=1`), coefficient of variation, minimum, maximum, and range. `cv` is a ratio; `cv_percent` is the same quantity multiplied by 100. For example, `cv = 0.112` and `cv_percent = 11.2` represent the same relative variation.

Use mean to describe typical performance, standard deviation and range to describe seed sensitivity, and CV only as a supporting relative measure—especially when the mean is close to zero.

In [7]:
summary = summarize_by_group(seed_runs)
key_metrics = [
    "best_val_acc",
    "final_val_acc",
    "best_val_loss",
    "final_acc_train_val_gap",
]
display(summary.loc[summary['metric'].isin(key_metrics)].reset_index(drop=True))

,experiment_group,metric,n,mean,std,cv,cv_percent,min,max,range
0,seed,final_val_acc,5,0.02190,0.002088,0.095345,9.534526,0.0200,0.0244,0.0044
1,seed,best_val_acc,5,0.02488,0.002791,0.112195,11.219515,0.0213,0.0275,0.0062
2,seed,best_val_loss,5,5.66892,0.013441,0.002371,0.237104,5.6584,5.6918,0.0334
3,seed,final_acc_train_val_gap,5,0.96448,0.003675,0.003810,0.380983,0.9593,0.9676,0.0083


### Human-readable best validation accuracy

The source accuracy remains on the 0–1 scale in saved CSV files. This view converts accuracy, standard deviation, and range to percentage points for reporting.

In [8]:
best_val_report = summary.loc[summary['metric'] == 'best_val_acc'].copy()
best_val_report['mean_percent'] = best_val_report['mean'] * 100
best_val_report['std_percentage_points'] = best_val_report['std'] * 100
best_val_report['min_percent'] = best_val_report['min'] * 100
best_val_report['max_percent'] = best_val_report['max'] * 100
best_val_report['range_percentage_points'] = best_val_report['range'] * 100
display(best_val_report[[
    'experiment_group', 'n', 'mean_percent', 'std_percentage_points',
    'cv_percent', 'min_percent', 'max_percent', 'range_percentage_points'
]].round(4))

,experiment_group,n,mean_percent,std_percentage_points,cv_percent,min_percent,max_percent,range_percentage_points
6,seed,5,2.488,0.2791,11.2195,2.13,2.75,0.62


## 6. Median-performing seed

For each group, select the observed run whose `best_val_acc` is closest to the group's median. If two runs are equally close, the smaller numeric seed is selected deterministically. This is a representative seed, not automatically the best seed for final reporting.

In [9]:
median_seeds = select_median_seed(seed_runs)
display(median_seeds)

,experiment_group,seed,best_val_acc,group_median_best_val_acc,distance_to_group_median
0,seed,0,0.0262,0.0262,0.0


## 7. Save tables and plots

The output directory will contain:

- `seed_runs_enriched.csv`: filtered run-level data plus train-validation gaps
- `seed_summary.csv`: long-form statistics for every metric
- `median_seed.csv`: one representative observed seed per group
- three PNG diagnostics for validation accuracy, train-vs-validation accuracy, and accuracy gaps

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
seed_runs.to_csv(OUTPUT_DIR / "seed_runs_enriched.csv", index=False)
summary.to_csv(OUTPUT_DIR / "seed_summary.csv", index=False)
median_seeds.to_csv(OUTPUT_DIR / "median_seed.csv", index=False)
plot_paths = plot_seed_analysis(seed_runs, OUTPUT_DIR)

print(f"Saved analysis to: {OUTPUT_RELATIVE}")
for path in plot_paths:
    print(f"- {path.name}")

Saved analysis to: results\analysis\seed
- best_val_acc_by_seed.png
- best_train_vs_val_acc.png
- accuracy_gaps_by_seed.png


## 8. Interpretation checklist

When comparing a new method with this seed baseline:

1. Report mean `best_val_acc` with sample standard deviation and min–max range.
2. Compare the method's improvement with the seed standard deviation; an improvement smaller than ordinary seed variation is weak evidence by itself.
3. Inspect final accuracy and the final train-validation gap for consistency and overfitting.
4. Use CV as supporting context, not the sole stability criterion when the metric mean is small.
5. Prefer repeated paired seeds and a suitable statistical test when making a formal method comparison.